<a href="https://colab.research.google.com/github/hongqin/Python-CoLab-bootcamp/blob/master/tinyAI_emoji_Chinese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# -*- coding: utf-8 -*-

import json
import re
import unicodedata
from pathlib import Path

# In Colab we'll persist learned words next to the notebook
LEXICON_PATH = Path("tinyai_lexicon.json")

# Regex for CJK characters
CJK_RE = re.compile(r"[\u4e00-\u9fff]")

def strip_accents(s: str) -> str:
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join([c for c in nfkd if not unicodedata.combining(c)])

def tokenize_multilang(text: str):
    """
    Tokenize English/Spanish into word tokens and Chinese into single characters,
    after lowercasing and accent stripping for Latin scripts.
    """
    text_l = text.lower()
    raw = re.findall(r"[A-Za-z0-9']+|[\u4e00-\u9fff]", text_l)
    return [strip_accents(tok) if not CJK_RE.match(tok) else tok for tok in raw]

def load_lexicon():
    # Base lexicon (English)
    base = {
        "love": 3, "like": 2, "happy": 2, "great": 2, "good": 1, "fun": 1, "yay": 2, "smile": 2,
        "sad": -2, "angry": -2, "mad": -2, "hate": -3, "bad": -1, "tired": -1, "bored": -1,
        # negations (used to flip the next sentiment word)
        "not": 0, "no": 0, "never": 0,
    }
    # Spanish (accent-stripped)
    base.update({
        "feliz": 2, "alegre": 2, "divertido": 1, "amor": 3, "bien": 1, "sonrisa": 2,
        "triste": -2, "enojado": -2, "furioso": -2, "cansado": -1, "aburrido": -1, "mal": -1
    })
    # Chinese bigrams & characters
    bigrams = {
        "开心": 3, "高兴": 2, "快乐": 3, "喜欢": 2, "爱你": 3,
        "伤心": -3, "难过": -3, "生气": -2, "讨厌": -3, "沮丧": -2, "累死": -2
    }
    chars = {
        "爱": 2, "好": 1, "喜": 1, "乐": 1, "笑": 2,
        "伤": -1, "恨": -3, "累": -1, "怒": -2, "烦": -1, "哭": -2
    }
    base["_zh_bigrams_"] = bigrams
    base["_zh_chars_"] = chars

    # Merge in any user-learned words from previous sessions
    if LEXICON_PATH.exists():
        try:
            learned = json.loads(LEXICON_PATH.read_text(encoding="utf-8"))
            for k, v in learned.items():
                base[k] = v
        except Exception:
            pass

    return base

def save_learned(lexicon):
    # Persist only simple numeric entries (skip helpers)
    reserved = {"_zh_bigrams_", "_zh_chars_"}
    out = {}
    for k, v in lexicon.items():
        if k in reserved:
            continue
        if isinstance(v, (int, float)):
            out[k] = v
    LEXICON_PATH.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

def score_sentence(tokens, lexicon):
    """
    Scoring order:
      1) Chinese bigrams (remove their chars from further scoring)
      2) Remaining Chinese single characters
      3) Latin tokens with simple negation (flip sign after not/no/never)
    Returns: (total_score, hits_list[(token, weight), ...])
    """
    zh_bi = lexicon.get("_zh_bigrams_", {})
    zh_ch = lexicon.get("_zh_chars_", {})

    # Build stream of Chinese chars to scan for bigrams
    chinese_stream = "".join([t for t in tokens if CJK_RE.match(t)])
    hits = []
    used_spans = []
    total = 0

    # 1) bigrams
    i = 0
    while i < len(chinese_stream) - 1:
        bg = chinese_stream[i:i+2]
        if bg in zh_bi:
            val = zh_bi[bg]
            total += val
            hits.append((bg, val))
            used_spans.append((i, i+2))
            i += 2
        else:
            i += 1

    # 2) single-character scoring (skip chars covered by bigrams)
    used_idx = set()
    for a, b in used_spans:
        used_idx.update(range(a, b))
    for idx, ch in enumerate(chinese_stream):
        if idx in used_idx:
            continue
        if ch in zh_ch:
            val = zh_ch[ch]
            total += val
            hits.append((ch, val))

    # 3) Latin tokens + simple negation
    negations = {"not", "no", "never"}
    flip_next = False
    for tok in tokens:
        if CJK_RE.match(tok):
            continue
        if tok in negations:
            flip_next = True
            continue
        val = lexicon.get(tok, 0)
        if val != 0:
            if flip_next:
                val = -val
                flip_next = False
            total += val
            hits.append((tok, val))

    return total, hits

def label_from_score(score: float):
    if score >= 2:
        return "positive", "🙂"
    if score <= -2:
        return "negative", "🙁"
    return "neutral", "😐"

def highlight_hits(tokens, hits):
    hit_set = {h[0] for h in hits}
    return " ".join([f"*{t}*" if t in hit_set else t for t in tokens])

def classify(text: str, lexicon=None):
    if lexicon is None:
        lexicon = load_lexicon()
    tokens = tokenize_multilang(text)
    score, hits = score_sentence(tokens, lexicon)
    label, emoji = label_from_score(score)
    return {"tokens": tokens, "score": score, "hits": hits, "label": label, "emoji": emoji}

def learn_from_feedback(tokens, prediction, lexicon):
    try:
        ans = input("Was the emoji right? (y/n) ").strip().lower()
    except EOFError:
        return
    if ans != "n":
        return

    map_mood = {"p": ("positive", +2), "n": ("negative", -2), "z": ("neutral", 0)}
    try:
        mood_in = input("Correct mood? [p]ositive / [z] neutral / [n]egative: ").strip().lower()[:1]
    except EOFError:
        return
    if mood_in not in map_mood:
        print("Skipping learning.")
        return
    _, base_val = map_mood[mood_in]

    try:
        teach = input("Pick a word/字符 from your sentence to teach (Enter to skip): ").strip()
    except EOFError:
        return
    if not teach:
        return

    if not CJK_RE.match(teach):
        teach = strip_accents(teach.lower())

    old = lexicon.get(teach, 0)
    new = max(-5, min(5, old + base_val))
    lexicon[teach] = new
    save_learned(lexicon)
    print(f"Learned: '{teach}' -> {new:+d} (was {old:+d}).")

def test_sentences(samples):
    lex = load_lexicon()
    for s in samples:
        result = classify(s, lex)
        print(f"Input: {s}")
        print(f"  Tokens: {result['tokens']}")
        print(f"  Score: {result['score']} | Label: {result['label']} {result['emoji']}")
        if result["hits"]:
            print("  Hits:", ", ".join([f"{w}({v:+d})" for w, v in result["hits"]]))
        else:
            print("  Hits: (none)")
        print()

def play():
    print("🧠 Tiny Emoji Mood Classifier — type a sentence; 'quit' to stop.\n")
    lex = load_lexicon()
    while True:
        try:
            s = input("> ").strip()
        except EOFError:
            break
        if s.lower() in {"quit", "exit"}:
            break
        result = classify(s, lex)
        print(f"{result['emoji']}  {result['label'].upper()} (score={result['score']})")
        if result["hits"]:
            print("   clues:", highlight_hits(result["tokens"], result["hits"]))
        learn_from_feedback(result["tokens"], result, lex)

# --- Example usage (run these manually in Colab) ---
# tests = [
#     "Estoy muy feliz, te amo",
#     "Hoy me siento triste",
#     "今天我很开心",
#     "我很累",
#     "I love my dog and sunny day",
#     "I am not happy about this",
# ]
# test_sentences(tests)
# play()

In [12]:
tests = [
    "Estoy muy feliz, te amo",
    "Hoy me siento triste",
    "今天我很开心",
    "我很累",
    "I love my dog and sunny day",
    "I am not happy about this",
]
test_sentences(tests)

Input: Estoy muy feliz, te amo
  Tokens: ['estoy', 'muy', 'feliz', 'te', 'amo']
  Score: 2 | Label: positive 🙂
  Hits: feliz(+2)

Input: Hoy me siento triste
  Tokens: ['hoy', 'me', 'siento', 'triste']
  Score: -2 | Label: negative 🙁
  Hits: triste(-2)

Input: 今天我很开心
  Tokens: ['今', '天', '我', '很', '开', '心']
  Score: 3 | Label: positive 🙂
  Hits: 开心(+3)

Input: 我很累
  Tokens: ['我', '很', '累']
  Score: -1 | Label: neutral 😐
  Hits: 累(-1)

Input: I love my dog and sunny day
  Tokens: ['i', 'love', 'my', 'dog', 'and', 'sunny', 'day']
  Score: 3 | Label: positive 🙂
  Hits: love(+3)

Input: I am not happy about this
  Tokens: ['i', 'am', 'not', 'happy', 'about', 'this']
  Score: -2 | Label: negative 🙁
  Hits: happy(-2)

